# A) Scrapping Data

In [ ]:
!pip install google-play-scraper # Corrected package name to google-play-scraper

In [ ]:
from google_play_scraper import app
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
#scrape jumlah ulasan yang diinginkan
from google_play_scraper import Sort, reviews

result, continuation_token = reviews(
    'com.gojek.app',
    lang='id', #scrape data ulasan gojek app yang ada di google play store
    country='id', #setting bahasa menjadi bindo
    sort=Sort.MOST_RELEVANT, #gunakan most_relevan untuk mendapatkan ulasan yang relevant
    count=1000, #jumlah ulasan yang diambil ada seribu
    filter_score_with=None #di filter_score gunakan none untuk mengambil semua score atau rating bintang 1 sampai 5
)

In [ ]:
#pengambilan kolom review
df_busu = pd.DataFrame(np.array(result),columns=['review'])

df_busu = df_busu.join(pd.DataFrame(df_busu.pop('review').tolist()))

df_busu.head()

In [ ]:
len(df_busu.index) #hitung kembali brp jumlah data yang didapat

In [ ]:
df_busu[['userName', 'at', 'content', 'score']].head()

In [ ]:
#urutkan data berdasarkan tanggal
new_df = df_busu[['userName', 'at', 'content', 'score']]
sorted_df = new_df.sort_values(by='at', ascending=False) #diurutkan dari terbaru (true kalo urutkan dari terlama)
sorted_df.head()

In [ ]:
my_df = sorted_df[['userName', 'at', 'content', 'score']] #setelah mendapatkan sun, tgl, ulasan, score  kemudian dapat disimpan ke variable my_df

In [ ]:
my_df.head()

In [ ]:
# Export the file to CSV
df_busu.to_csv('ulasangojek.csv', index=False)

# Download the file directly
files.download('ulasangojek.csv')

# B) Sentiment Analysis

# **1) Import Library**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle

%matplotlib inline

In [ ]:
# Install sastrawi untuk library bahasa Indonesia
!pip -q install sastrawi

In [ ]:
# Install untuk melakukan filter stopwords (kata-kata yang kurang penting)
import nltk
nltk.download('stopwords')

# **2) Data Acquisition**

In [ ]:
data = pd.read_csv('/content/ulasangojek.csv')
data

In [ ]:
data.info()

In [ ]:
data.isnull().sum().sum()

In [ ]:
data.dropna()

In [ ]:
# Kelompokkan score menjadi sentiment
def assign_sentiment(score):
    if score in [1, 2]:
        return 'Negatif'
    elif score == 3:
        return 'Netral'
    elif score in [4, 5]:
        return 'Positif'

In [ ]:
# Tambahkan kolom baru di data berdasarkan fungsi sentiment
data['sentimen'] = data['score'].apply(assign_sentiment)

In [ ]:
height = data['sentimen'].value_counts()
labels = ('Negatif', 'Netral', 'Positif')

x_pos = np.arange(len(labels))
y_pos = np.arange(len(labels))

plt.figure(figsize=(7, 4), dpi=80)
plt.ylim(0,300)
plt.title('Distribusi Sentimen Ulasan Aplikasi Gojek', fontweight='bold')
plt.xlabel('Kategori', fontweight='bold')
plt.ylabel('Jumlah', fontweight='bold')
plt.bar(y_pos, height, color='skyblue')
plt.xticks(y_pos, labels)
plt.show()

# **3) Data Pre-processing**

## Case Folding

Proses untuk mengubah semua teks di dalam dataset menjadi huruf kecil agar data bisa jadi lebih konsisten.


In [ ]:
import re

In [ ]:
# Buat fungsi untuk langkah case folding
def casefolding(text):
    text = text.lower() # Mengubah teks jadi lower case
    text = re.sub(r'http\S+', '', text) # Menghapus URL
    text = re.sub(r'\d+', '', text) # Menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # Menghapus karakter tanda baca
    text = text.strip() # Menghapus spasi di awal dan akhir teks
    return text

In [ ]:
raw_sample = data['content'].iloc[1]
case_folding = casefolding(raw_sample)

print(f'Raw: {raw_sample}')
print(f'Case folding: {case_folding}')

## Normalization

Proses untuk memodifikasi nilai dalam variabel sehingga kita dapat mengukurnya dalam skala umum.

In [ ]:
# Kumpulan slangwords
!wget https://raw.githubusercontent.com/ksnugroho/klasifikasi-spam-sms/master/data/key_norm.csv

In [ ]:
key_norm = pd.read_csv('/content/key_norm.csv')
print(key_norm.shape)
key_norm.head()

In [ ]:
# Buat fungsi untuk langkah text normalization
def text_normalize(text):
    text = ' '.join([key_norm.loc[key_norm['singkat'] == word, 'hasil'].values[0] if (key_norm['singkat'] == word).any() else word for word in text.split()])
    text = str.lower(text)
    return text

In [ ]:
raw_sample = data['content'].iloc[11]
case_folding = casefolding(raw_sample)
text_normalization = text_normalize(case_folding)

print(f'Raw: {raw_sample}')
print(f'Case folding: {case_folding}')
print(f'Text normalization: {text_normalization}')

## Stopword Removal

Proses menghilangkan kata-kata umum yang sering muncul dan tidak memberikan informasi penting.

In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

stopwords_ind = stopwords.words('indonesian')

In [ ]:
# Buat fungsi untuk langkah stopword removal
def stopword_removal(text):
    clean_words = []
    text = text.split()
    for word in text:
        if word not in stopwords_ind:
            clean_words.append(word)
    return " ".join(clean_words)
    return text

In [ ]:
raw_sample = data['content'].iloc[5]
case_folding = casefolding(raw_sample)
stopword_removal = stopword_removal(case_folding)

print(f'Kalimat asli: {raw_sample}')
print(f'Kalimat setelah stopword removal: {stopword_removal}')

## Stemming

Proses menghilangkan imbuhan baik yang berada di awal kata ataupun yang berada di akhir.

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

In [ ]:
# Buat fungsi untuk langkah stemming bahasa Indonesia
def stemming(text):
    text = stemmer.stem(text)
    return text

In [ ]:
raw_sample = data['content'].iloc[8]
case_folding = casefolding(raw_sample)
normalize = text_normalize(case_folding)
stopword_removal = stopword_removal(normalize)
text_stemming = stemming(stopword_removal)

print(f'Kalimat asli: {raw_sample}')
print(f'Kalimat setelah case folding: {case_folding}')
print(f'Kalimat setelah text normalization: {normalize}')
print(f'Kalimat setelah stopword removal: {stopword_removal}')
print(f'Kalimat setelah stemming: {text_stemming}')

## Text Preprocessing Pipeline

In [ ]:
# Buat fungsi untuk menggabungkan seluruh langkah data preprocessing
def text_preprocessing(text):
    text = casefolding(text)
    text = text_normalize(text)
    text = stopword_removal(text)
    text = stemming(text)
    return text

In [ ]:
%%time
data['clean_content'] = data['content'].apply(text_preprocessing)

In [ ]:
data

# **4) Feature Engineering**

In [ ]:
# Pisahkan kolom sentimen dan clean data
x = data['clean_content']
y = data['sentimen']

In [ ]:
x

In [ ]:
y

## Feature Extraction (TF-IDF dan N-Gram)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf_idf = TfidfVectorizer(ngram_range=(1,1))
tf_idf.fit(x)

In [ ]:
# Melihat unique words
print(len(tf_idf.get_feature_names_out()))

In [ ]:
# Melihat unique words yang ada dalam corpus
print(tf_idf.get_feature_names_out())

In [ ]:
# Melakukan transformasi teks ke matriks TF-IDF
x_tf_idf = tf_idf.transform(x).toarray()
x_tf_idf

In [ ]:
# Membuat DataFrame dari Matriks TF-IDF
data_tf_idf = pd.DataFrame(x_tf_idf, columns=tf_idf.get_feature_names_out())
data_tf_idf

In [ ]:
# Menyimpan Matriks TF-IDF ke File
with open('tf_idf_feature.pickle', 'wb') as output:
    pickle.dump(x_tf_idf, output)

# Data ini siap untuk dimasukkan dalam proses ML

# 5) Machine Learning Modeling

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from joblib import dump

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_tf_idf, y, test_size=0.2, random_state=50)

## 1 - Support Vector Machine

In [ ]:
svm_algorithm = SVC
svm_model = svm_algorithm().fit(x_train, y_train)

dump(svm_model, filename='svm_model.joblib')

In [ ]:
# Gunakan model yang telah di-train untuk memprediksi label
model_predict1  = svm_model.predict(x_test)

# Tampilkan hasil prediksi
model_predict1

In [ ]:
# Tampilkan label sebenarnya pada data yang diuji
y_test

In [ ]:
# Hitung jumlah data yang berhasil di prediksi model & jumlah data yang salah di prediksi
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score

# Melakukan cross validation
cv = ShuffleSplit(n_splits=10, test_size=0.2, random_state=50)

cv_accuracy = (cross_val_score(svm_model, x_tf_idf, y, cv=cv, scoring='accuracy'))
avg_accuracy = np.mean(cv_accuracy)

# Menghitung prediksi benar-salah
prediksi_benar = (model_predict1 == y_test).sum()
prediksi_salah = (model_predict1 != y_test).sum()

print('Model Support Vector Machine')
print('Jumlah prediksi benar:', prediksi_benar)
print('Jumlah prediksi salah:', prediksi_salah)

# Menghitung Akurasi Pengujian
accuracy = prediksi_benar / (prediksi_benar + prediksi_salah) * 100
print('Akurasi pengujian:', accuracy, '%\n')

print('Akurasi setiap split:', cv_accuracy)
print('Rata-rata akurasi pada cross validation:', avg_accuracy)

## 2 - Naive Bayes

In [ ]:
nb_algorithm = MultinomialNB
nb_model = nb_algorithm().fit(x_train, y_train)

dump(nb_model, filename='nb_model.joblib')

In [ ]:
# Gunakan model yang telah di-train untuk memprediksi label
model_predict2 = nb_model.predict(x_test)

# Tampilkan hasil prediksi
model_predict2

In [ ]:
# Tampilkan label sebenarnya pada data yang diuji
y_test

In [ ]:
# Hitung jumlah data yang berhasil di prediksi model & jumlah data yang salah di prediksi
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score

# Melakukan cross validation
cv = ShuffleSplit(n_splits=10, test_size=0.2, random_state=50)

cv_accuracy = (cross_val_score(nb_model, x_tf_idf, y, cv=cv, scoring='accuracy'))
avg_accuracy = np.mean(cv_accuracy)

# Menghitung prediksi benar salah
prediksi_benar = (model_predict2 == y_test).sum()
prediksi_salah = (model_predict2 != y_test).sum()

print('Model Naive Bayes')
print('Jumlah prediksi benar:', prediksi_benar)
print('Jumlah prediksi salah:', prediksi_salah)

# Menghitung Akurasi Pengujian
accuracy = prediksi_benar / (prediksi_benar + prediksi_salah) * 100
print('Akurasi pengujian:', accuracy, '%\n')

print('Akurasi setiap split:', cv_accuracy)
print('Rata-rata akurasi pada cross validation:', avg_accuracy)